In [1]:
import pandas as pd
import glob
import os
import numpy as np

from classify import classify_file

folder_path = "./pipeline_steps/input_files/raw"

/home/afunnell/miniconda3/envs/pipeline_new/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
xlsx_files = glob.glob(os.path.join(folder_path, "*.xlsx"))

all_files = csv_files + xlsx_files

dfs = []

for file in all_files:
    if file.endswith(".csv"):
        df = pd.read_csv(file)
    elif file.endswith(".xlsx"):
        df = pd.read_excel(file)
    else:
        continue
    
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)

In [5]:
# your mapping dict (canonical -> list of possible source columns)
COL_MAP = {
    "CaseNumber": ["CaseNumber", "CaseNum", "Case Number", "Case#"],
    "ResidenceType": ["ResidenceType", "ResType", "Residence Type"],
    "DeathDate": ["DeathDate", "Date of Death", "Death Date", "DateOfDeath", "DateofDeath"],
    "DeathTime": ["DeathTime", "Time of Death", "Death Time", "TimeofDeath"],
    "DeathAddress": ["DeathAddress", "DeathAddr", "DeathAdress", "DeathAddr.1", "address.death", "DeathAdress.1", "Death Address"],
    "DeathZip": ["DeathZip", "DeathZip.1", "DeathZipCode", "DeathZi\np", "Zip"],
    "DeathCity": ["DeathCity", "Death City", "DeathCityDesc"],
    "EventPlace": ["EventPlace", "Event Place"],
    "EventAddress": ["EventAddress", "EventAddr", "EventAddr.1", "eventaddress", "Event Address"],
    "EventZip": ["EventZip", "EventZip.1", "EventZi\np", "EventZipCode", "Zip.1", "Event Zip"],
    "EventCity": ["EventCity", "EventCityDesc"],
    "Mode": ["Mode", "Mode.1"],
    "CauseA": ["CauseA", "Cause A", "DeathCauseA"],
    "CauseB": ["CauseB", "Cause B", "DeathCauseB"],
    "CauseC": ["CauseC", "Cause C", "DeathCauseC"],
    "CauseD": ["CauseD", "Cause D", "DeathCauseD"],
    "CauseOther": ["CauseOther", "Other Cause", "OtherCause"],
    "HowInjuryOccurred": ["HowInjuryOccurred", "InjuryDesc", "HowInjuryOccu\nrred"],
    "FirstName": ["FirstName", "First Name"],
    "MiddleName": ["MiddleName", "Middle Name"],
    "LastName": ["LastName", "Last Name"],
    "DateofBirth": ["DateofBirth", "Date of Birth", "BirthDate"],
    "Text": ["Text", "text"],
    "Address": ["Address", "address"],
    "Race": ["Race", "Races"],
}

def _blank_to_nan(s: pd.Series) -> pd.Series:
    """Treat empty/whitespace strings as missing."""
    if s.dtype == "object":
        s = s.replace(r"^\s*$", np.nan, regex=True)
    return s

def coalesce_columns(df: pd.DataFrame, col_map: dict[str, list[str]], drop_sources: bool = False) -> pd.DataFrame:
    df = df.copy()

    for canon, aliases in col_map.items():
        # keep only aliases that actually exist in df
        present = [c for c in aliases if c in df.columns]
        if not present:
            continue

        # start with an all-missing series
        out = pd.Series(np.nan, index=df.index)

        # fill from each alias in order (first non-missing wins)
        for c in present:
            src = _blank_to_nan(df[c])
            out = out.combine_first(src)

        df[canon] = out

        if drop_sources:
            # don't drop the canonical column if it was also an alias name
            to_drop = [c for c in present if c != canon]
            df = df.drop(columns=to_drop, errors="ignore")

    return df



def coalesce_duplicate_rows(df, subset, tie_keep="first"):
    """
    Collapse rows that share the same value(s) in `subset` into a single row.

    Instead of arbitrarily keeping the first/last duplicate, the row with the
    most non-null values is used as the base, and any remaining nulls are then
    filled in using values from the other rows in the same group.

    Parameters:
    - df (pd.DataFrame): The DataFrame to deduplicate.
    - subset (str or list): Column name(s) that identify a duplicate.
    - tie_keep (str): Which row wins for cells where equally complete rows
      hold *different* non-null values. "first" keeps the earlier row's
      value, "last" keeps the later row's value (mirrors drop_duplicates'
      keep argument).

    Returns:
    - pd.DataFrame: One row per unique `subset` value, with nulls minimized.
    """
    original_columns = df.columns.tolist()

    # Rank rows by how complete they are (most non-null values first).
    completeness = df.notna().sum(axis=1)
    df_ranked = df.assign(_completeness=completeness)

    # Among equally complete rows a *stable* sort preserves the existing row
    # order, so the first row would win ties. Reverse first when the caller
    # wants the last row to win instead.
    if tie_keep == "last":
        df_ranked = df_ranked[::-1]

    df_sorted = df_ranked.sort_values(
        "_completeness", ascending=False, kind="stable"
    ).drop(columns="_completeness")

    # groupby().first() takes the first *non-null* value in each column, so
    # the base (most complete) row's values win and any gaps are filled from
    # the remaining rows in the group.
    combined = df_sorted.groupby(
        subset, as_index=False, sort=False, dropna=False
    ).first()

    # Restore the original column order (groupby moves the key columns first).
    return combined[original_columns]



def coalesce_duplicate_rows(df, subset, tie_keep="first"):
    """
    Collapse rows that share the same value(s) in `subset` into a single row.

    Instead of arbitrarily keeping the first/last duplicate, the row with the
    most non-null values is used as the base, and any remaining nulls are then
    filled in using values from the other rows in the same group.

    Parameters:
    - df (pd.DataFrame): The DataFrame to deduplicate.
    - subset (str or list): Column name(s) that identify a duplicate.
    - tie_keep (str): Which row wins for cells where equally complete rows
      hold *different* non-null values. "first" keeps the earlier row's
      value, "last" keeps the later row's value (mirrors drop_duplicates'
      keep argument).

    Returns:
    - pd.DataFrame: One row per unique `subset` value, with nulls minimized.
    """
    original_columns = df.columns.tolist()

    # Rank rows by how complete they are (most non-null values first).
    completeness = df.notna().sum(axis=1)
    df_ranked = df.assign(_completeness=completeness)

    # Among equally complete rows a *stable* sort preserves the existing row
    # order, so the first row would win ties. Reverse first when the caller
    # wants the last row to win instead.
    if tie_keep == "last":
        df_ranked = df_ranked[::-1]

    df_sorted = df_ranked.sort_values(
        "_completeness", ascending=False, kind="stable"
    ).drop(columns="_completeness")

    # groupby().first() takes the first *non-null* value in each column, so
    # the base (most complete) row's values win and any gaps are filled from
    # the remaining rows in the group.
    combined = df_sorted.groupby(
        subset, as_index=False, sort=False, dropna=False
    ).first()

    # Restore the original column order (groupby moves the key columns first).
    return combined[original_columns]

combined_df = coalesce_columns(combined_df, COL_MAP, drop_sources=True)  

/tmp/ipykernel_3164961/2968612090.py:51: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  out = out.combine_first(src)


In [6]:
duplicates_simple_drop = combined_df.drop_duplicates(subset='CaseNumber', keep="first")

In [7]:
combined_df = coalesce_duplicate_rows(combined_df, 'CaseNumber')

In [8]:
combined_df.isna().sum()

Age                            217
Gender                          23
Race                          1391
DeathDate                        2
DeathPlace                     723
DeathAddress                  4027
DeathCity                     5505
DeathZip                     36701
EventPlace                   63740
EventCity                    76014
EventZip                     77226
Mode                          1593
CauseA                        1574
CauseB                      120430
CauseC                      141255
CauseD                      145093
CauseOther                   80579
HowInjuryOccurred            66817
FirstName                     4100
LastName                      4057
EventAddress                 68838
SourcePages                  34233
ResidenceType                19791
Death Zip Code               56157
CaseNumber                       0
DeathTime                    29371
ExperiencingHomelessness    142679
MiddleName                   70232
DateofBirth         

In [9]:
combined_df[combined_df['CaseNumber'] == "2017-01320"]

,Age,Gender,Race,DeathDate,DeathPlace,DeathAddress,DeathCity,DeathZip,EventPlace,EventCity,...,LastName,EventAddress,SourcePages,ResidenceType,Death Zip Code,CaseNumber,DeathTime,ExperiencingHomelessness,MiddleName,DateofBirth
146040,None,UNKNOWN,None,2/15/2017,None,None,None,90247,None,None,...,None,None,"3408,3409,3410,3411,3412,3413,3414,3415,3416,3...",None,NaN,2017-01320,None,NaN,None,NaT


In [10]:
duplicates_simple_drop[duplicates_simple_drop['CaseNumber'] == "2017-01320"]

,Age,Gender,Race,DeathDate,DeathPlace,DeathAddress,DeathCity,DeathZip,EventPlace,EventCity,...,LastName,EventAddress,SourcePages,ResidenceType,Death Zip Code,CaseNumber,DeathTime,ExperiencingHomelessness,MiddleName,DateofBirth
74531,NaN,UNKNOWN,NaN,2/15/2017,NaN,NaN,NaN,90247,NaN,NaN,...,NaN,NaN,"3408,3409,3410,3411,3412,3413,3414,3415,3416,3...",NaN,NaN,2017-01320,NaN,NaN,NaN,NaT


In [11]:
s = combined_df["DeathDate"]

# Pass 1: parse as-is (won't raise; unparseable rows -> NaT)
death_dt = pd.to_datetime(s, errors="coerce")

# Pass 2: only fix the failures by extracting a date/datetime token
mask = death_dt.isna() & s.notna()

s_bad = s.loc[mask].astype("string")

# Extract first date/datetime-looking token (handles "Facility 5/8/2024", "2022-12-05 00:00:00\tRESIDENCE ...", etc.)
token = s_bad.str.extract(
    r'('
    r'\d{4}-\d{2}-\d{2}(?:[ T]\d{2}:\d{2}:\d{2})?'   # 2022-12-05 or 2022-12-05 00:00:00
    r'|'
    r'\d{1,2}/\d{1,2}/\d{2,4}'                       # 6/1/2023 or 5/8/2024
    r'|'
    r'\d{1,2}-\d{1,2}-\d{2,4}'                       # 6-1-2023
    r')',
    expand=False
)

death_dt.loc[mask] = pd.to_datetime(token, errors="coerce")

combined_df["DeathDate_parsed"] = death_dt

In [12]:
combined_df['DeathDate'] = combined_df['DeathDate_parsed']

In [13]:
combined_df['DeathDate'] = pd.to_datetime(combined_df['DeathDate'], format="mixed")

In [14]:
combined_df = combined_df.sort_values("DeathDate")

In [15]:
combined_df = combined_df.drop(columns=['DeathDate_parsed'])

In [16]:
combined_df = combined_df.drop_duplicates(subset='CaseNumber', keep="first")

In [17]:

def clean_and_categorize_race(race):
    if pd.isna(race):
        return np.nan

    race = race.lower()           # Case folding
    race = race.replace(" ", "")  # Remove spaces
    race = race.replace("\n", "") # Remove newline characters

    # If multiple races are listed in one row, pick the first
    if "," in race:
        parts = [p for p in race.split(",") if p not in ("unknown/other", "unknown", "null")]
        race = parts[0] if parts else "unknown/other"
        
    if race in ["americanindian", "nativeamerican"]:
        return "AMERICAN INDIAN"
    elif race in ["armenian", "middleeastern"]:
        return "MIDDLE EASTERN"
    elif race in [
        "asian",
        "cambodian",
        "chinese",
        "filipino",
        "japanese",
        "korean",
        "eastindian",
        "thai",
        "vietnamese",
    ]:
        return "ASIAN"
    elif race in ["black"]:
        return "BLACK"
    elif race in [
        "guamanian",
        "hawaiian",
        "pacificislander",
        "samoan",
        "tongan",
        "nativehawaiian/otherpacificislander",
    ]:
        return "PACIFIC ISLANDER"
    elif race in ["hispanic/latino", "hispanic/latina", "hispanic/latinamerican"]:
        return "LATINE"
    elif race in ["white", "caucasian", "white/caucasian"]:
        return "WHITE"
    elif race in ["unknown", "null", "unknown/other"]:
        return np.nan
    else:
        return race
    
def final_clean(current_df):


    valid_races = [
    "WHITE",
    "LATINE",
    "BLACK",
    "ASIAN",
    "MIDDLE EASTERN",
    "AMERICAN INDIAN",
    "PACIFIC ISLANDER",
    "UNKNOWN"
    ]


    gender_map = {
        "M": "MALE",
        "MALE": "MALE",
        "F": "FEMALE",
        "FEMALE": "FEMALE",
        "NON-BINARY": "NON-BINARY"
    }

    current_df["Mode"] = (
        current_df["Mode"]
        .str.replace("\n", "", regex=False)   
        .str.strip()                          
        .str.upper()                          
    )

    current_df["Mode"] = current_df["Mode"].replace({
        "UNDETERMI": "UNDETERMINED"
    })

    # standardize then filter
    current_df["Race"] = (
        current_df["Race"]
        .str.replace("\n", "", regex=False)
        .str.strip()
        .str.upper()
    )

    current_df.loc[~current_df["Race"].isin(valid_races), "Race"] = np.nan

    current_df["Gender"] = (
        current_df["Gender"]
        .str.replace("\n", "", regex=False)
        .str.strip()
        .str.upper()
    )


    current_df["Gender"] = current_df["Gender"].map(gender_map)
    
    return current_df



In [18]:
combined_df['Race'] = combined_df['Race'].apply(clean_and_categorize_race)

In [19]:
combined_df = final_clean(combined_df)

In [24]:
combined_df.columns

Index(['Age', 'Gender', 'Race', 'DeathDate', 'DeathPlace', 'DeathAddress',
       'DeathCity', 'DeathZip', 'EventPlace', 'EventCity', 'EventZip', 'Mode',
       'CauseA', 'CauseB', 'CauseC', 'CauseD', 'CauseOther',
       'HowInjuryOccurred', 'FirstName', 'LastName', 'EventAddress',
       'SourcePages', 'ResidenceType', 'Death Zip Code', 'CaseNumber',
       'DeathTime', 'ExperiencingHomelessness', 'MiddleName', 'DateofBirth'],
      dtype='object')

In [ ]:
combined_df[combined_df['CaseNumber'] == "2023-16906"]

KeyError: 'source'

In [ ]:
MODEL_NAME = "bert_models/bioclinicalbert"

classify_file(combined_df, "classified_all_deaths_06012026_regex.csv", MODEL_NAME)

In [ ]:
all_deaths = pd.read_csv("./classified_all_deaths_06012026_regex.csv")